# Testing the non-standard (entity) flows

One cell per Step-0 entity flow. Edit the parameters at the top of a cell, run
it, and it drives the **real** `/query_search` streaming pipeline in-process
(`stream_full_pipeline`) with a query crafted to trigger that flow.

Each run emits one trace. View it in **Grafana → Tempo**: find the
`query_search.branch` span for the flow and read its `branch_*` attributes
(`branch_entities`, `branch_entity_resolved_counts`, `branch_result_count`, the
per-flow aliases / tiers / weave-seats / fallback flags), the per-person
resolution child spans (person), the two resolution child spans
(character_franchise), and the Qdrant span (similarity). A zero-result flow
fires an `entity flow empty` span event.

**The qualifier trap:** Step 0 routes to an entity flow only for a bare entity
name (+ optional "movies"/"films"). Any real qualifier — a genre, era, mood,
"directed by", "but funnier" — forces `none_of_the_above` (the standard flow).
Keep the parameters to bare names/titles. Each cell prints the fetches that
fired and flags a mis-route.

> Run **Cell 0 (setup)** and **Cell 1 (runner)** first; they must succeed before
> the per-flow cells.

In [1]:
# Cell 0 — setup (idempotent; safe to re-run).
# Opens Postgres/Redis/Qdrant and configures OpenTelemetry so the spans this
# notebook generates EXPORT to your local otel-lgtm / Tempo. Spans only reach
# Tempo if that container is running (grafana/otel-lgtm on :4317). Without it,
# everything still runs — the exporter just drops spans in the background.
import sys, time
from pathlib import Path

# Project root on sys.path (handles running from search_v2/).
project_root = str(Path.cwd().parent) if Path.cwd().name == "search_v2" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(Path(project_root) / ".env")

# Configure tracing FIRST (before opening the pool) so psycopg/redis/httpx are
# patched before any connection is made. setup_tracing needs a FastAPI app to
# instrument; a throwaway one is fine — we only want the global tracer provider
# + OTLP exporter it installs. Idempotent via its own _configured guard.
from fastapi import FastAPI
from observability.tracing import setup_tracing
setup_tracing(FastAPI())
print("Tracing:  configured (exports to OTEL_EXPORTER_OTLP_ENDPOINT if otel-lgtm is up)")

from db.postgres import pool as postgres_pool
from db.qdrant import check_qdrant
from db.redis import init_redis, check_redis
import db.redis as _redis_module

if getattr(postgres_pool, "_closed", True):
    await postgres_pool.open()
    print("Postgres: pool opened")
else:
    print("Postgres: pool already open")

if _redis_module._redis_client is None:
    await init_redis()
    print(f"Redis:    initialized ({await check_redis()})")
else:
    print(f"Redis:    already initialized ({await check_redis()})")

print(f"Qdrant:   {await check_qdrant()}")


Tracing:  configured (exports to OTEL_EXPORTER_OTLP_ENDPOINT if otel-lgtm is up)
Postgres: pool opened
Redis:    initialized (ok)
Qdrant:   ok


In [2]:
# Cell 1 — shared runner. Drives the REAL /query_search streaming pipeline
# (search_v2.streaming_orchestrator.stream_full_pipeline) in-process, prints the
# SSE-shaped events, and confirms the intended entity flow actually fired (so a
# Step-0 mis-route is obvious). Each run produces one trace: open it in Grafana
# → Tempo and inspect the `query_search.branch` span for that flow's `branch_*`
# attributes (entities, resolved counts, aliases, tiers, weave seats, etc.).
import time
from search_v2.streaming_orchestrator import stream_full_pipeline


async def run_flow(query: str, *, expected_type: str, clarification: str | None = None):
    print(f"QUERY: {query!r}   (expecting entity flow: {expected_type})")
    fired_types: list[str] = []
    async for event_name, payload in stream_full_pipeline(
        query, clarification=clarification, metadata_filters=None
    ):
        if event_name == "fetches_ready":
            fired_types = [f["type"] for f in payload["fetches"]]
            print("  fetches_ready:", fired_types)
        elif event_name == "branch_results":
            n = len(payload.get("results") or [])
            err = payload.get("branch_error")
            line = f"  branch_results[{payload['fetch_id']}]: {n} cards"
            print(line + (f"   BRANCH_ERROR={err}" if err else ""))
        elif event_name == "error":
            print("  ERROR event:", payload)
        elif event_name == "done":
            print(f"  done in {payload['total_elapsed']:.2f}s")
    ok = expected_type in fired_types
    verdict = "OK" if ok else "MISROUTED"
    print(f"  [{verdict}] expected '{expected_type}' among {fired_types}\n")
    return fired_types


/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## person

In [4]:
# --- person flow ---
# Bare person name(s), any credited role. A qualifier ("directed by", a genre,
# an era) would force none_of_the_above, so keep it to the name(s).
PEOPLE = ["Tom Hanks", "Tom Holland", "Brad Pit", "The Rock"]                       # e.g. ["Tom Hanks", "Woody Harrelson"]
query = " and ".join(PEOPLE) if len(PEOPLE) > 1 else PEOPLE[0]
await run_flow(query, expected_type="person")


QUERY: 'Tom Hanks and Tom Holland and Brad Pit and The Rock'   (expecting entity flow: person)
  fetches_ready: ['person']
  branch_results[person]: 100 cards
  done in 2.76s
  [OK] expected 'person' among ['person']



['person']

## similarity

In [3]:
# --- similarity flow ---
# An explicit "movies like ..." frame, or a bare list of >=2 titles. >=2 titles
# exercises the multi-anchor path (weave seats + low-cohesion fallback signals);
# a single title exercises single-anchor.
TITLES = ["Avengers Ultron", "Infinity War"]       # try 1 title for the single-anchor path
query = "movies like " + ", ".join(TITLES)
await run_flow(query, expected_type="similarity")


QUERY: 'movies like Avengers Ultron, Infinity War'   (expecting entity flow: similarity)
  fetches_ready: ['similarity']
  branch_results[similarity]: 50 cards
  done in 10.59s
  [OK] expected 'similarity' among ['similarity']



['similarity']

## exact_title

In [16]:
# --- exact_title flow ---
# The typed span must equal a real canonical title (NO plural "movies"/"films",
# which would make it a descriptor query). A stated year is captured separately
# by Step 0 (branch_exact_title_year) and enables the title_only source count.
TITLE = "Phantom of the Opera"
YEAR = 2004                                   # set to None to omit the year
query = f"{TITLE} {YEAR}" if YEAR else TITLE
await run_flow(query, expected_type="exact_title")


QUERY: 'Phantom of the Opera 2004'   (expecting entity flow: exact_title)
  fetches_ready: ['exact_title']
  branch_results[exact_title]: 10 cards
  done in 2.78s
  [OK] expected 'exact_title' among ['exact_title']



['exact_title']

## studio

In [14]:
# --- studio flow ---
# Studio / production-company name(s). Prefer unambiguous names (e.g. "A24" over
# "Up"). Multiple => union ("A24 and Neon").
STUDIOS = ["A24", "Neon", "LAIKA"]                             # e.g. ["A24", "Neon"]
query = " and ".join(STUDIOS)
await run_flow(query, expected_type="studio")


QUERY: 'A24 and Neon and LAIKA'   (expecting entity flow: studio)
  fetches_ready: ['studio']
  branch_results[studio]: 100 cards
  done in 8.36s
  [OK] expected 'studio' among ['studio']



['studio']

## non_character_franchise

In [19]:
# --- non_character_franchise flow ---
# A series / IP umbrella with disjoint protagonists. Abbreviations ("MCU") are
# bridged downstream by the LLM alias expansion (branch_aliases). Single entity.
FRANCHISE = "LOTR"                        # e.g. "The Lord of the Rings", "MCU"
await run_flow(FRANCHISE, expected_type="non_character_franchise")


QUERY: 'LOTR'   (expecting entity flow: non_character_franchise)
  fetches_ready: ['non_character_franchise']
  branch_results[non_character_franchise]: 11 cards
  done in 3.20s
  [OK] expected 'non_character_franchise' among ['non_character_franchise']



['non_character_franchise']

## character_franchise

In [20]:
# --- character_franchise flow ---
# A fictional protagonist persisting across films (different actors OK). Single
# entity; alias expansion produces both character_forms and franchise_forms.
CHARACTER = "Spider-Man"                        # e.g. "James Bond", "Batman"
await run_flow(CHARACTER, expected_type="character_franchise")


QUERY: 'Spider-Man'   (expecting entity flow: character_franchise)
  fetches_ready: ['character_franchise']
  branch_results[character_franchise]: 27 cards
  done in 4.38s
  [OK] expected 'character_franchise' among ['character_franchise']



['character_franchise']